##DBTC-1318

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.collateral
ADD COLUMN CollateralStatusChangeDate date AFTER CollateralStatus;

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.collateral
ADD COLUMN CollateralREStatusChangeDate date AFTER CollateralStatusChangeDate;

##DD-187 
Addition of new field ProductCategory in Product

In [0]:
env="p"

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.product 
ADD COLUMNS  ProductCategory string AFTER BondedFlag;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryarchive.product 
ADD COLUMNS  ProductCategory string AFTER BondedFlag;
""")

In [0]:

spark.sql(f"""
merge into `{env}-cdh`.primaryactive.product target
using `{env}-cdh`.landed.qc_subproduct src
on src.SPRD_CODE = target.ProductCode
and target.SourceSystem='qc'
and src.BusinessDate=target.BusinessDate
when matched then 
update set target.ProductCategory=src.PRD_DESC;
""")

##DATA2-297
###Steps
* Run the following code
* Re-run assignment_qca for all businessdates in prod.

In [0]:
%sql
DELETE FROM `p-cdh`.landed.qca_case_assignment a
WHERE EXISTS (
  SELECT 1
  FROM `p-de-sandbox`.temp.assignment_domus_historical h
  WHERE a.CASE_CODE = h.CASE_CODE
    AND a.BusinessDate = h.BusinessDate
);


DELETE FROM `p-cdh`.landed.qca_case_assignment a
WHERE EXISTS (
  SELECT 1
  FROM `p-de-sandbox`.temp.assignment_monza_historical h
  WHERE a.CASE_CODE = h.CASE_CODE
    AND a.BusinessDate = h.BusinessDate
);


DELETE FROM `p-cdh`.landed.qca_case_assignment a
WHERE EXISTS (
  SELECT 1
  FROM `p-de-sandbox`.temp.assignment_rhodium_historical h
  WHERE a.CASE_CODE = h.CASE_CODE
    AND a.BusinessDate = h.BusinessDate
);

In [0]:
%sql
insert into `p-cdh`.landed.qca_case_assignment
(CASE_CODE, CASE_ACCOUNT_FOR_SEARCH,    CUST_CODE,  AssignmentChannel,  ExternalCompanyAssignment,  
ExternalCompanyName,    AssignmentTeam, AssignmentAgent,    AssignmentAgentLoginName,   Branch, 
BusinessDate,   LoadingDateTime,    AssignmentProcess,  portfolio_id_qca)

SELECT 
 CASE_CODE, CASE_ACCOUNT_FOR_SEARCH ,CUST_CODE  ,AssignmentChannel, ExternalCompanyAssignment   
 ,ExternalCompanyName,  AssignmentTeam, AssignmentAgent,    AssignmentAgentLoginName,   Branch, 
 BusinessDate , getdate(), AssignmentProcess,    SPET_CODE
  FROM `p-de-sandbox`.temp.assignment_domus_historical 

union

SELECT 
 CASE_CODE, CASE_ACCOUNT_FOR_SEARCH ,CUST_CODE  ,AssignmentChannel, ExternalCompanyAssignment   
 ,ExternalCompanyName,  AssignmentTeam, AssignmentAgent,    AssignmentAgentLoginName,   Branch, 
 BusinessDate , getdate(), AssignmentProcess,    SPET_CODE
  FROM `p-de-sandbox`.temp.assignment_rhodium_historical 

union 

SELECT 
 CASE_CODE, CASE_ACCOUNT_FOR_SEARCH ,CUST_CODE  ,AssignmentChannel, ExternalCompanyAssignment   
 ,ExternalCompanyName,  AssignmentTeam, AssignmentAgent,    AssignmentAgentLoginName,   Branch, 
 BusinessDate , getdate(), AssignmentProcess,    SPET_CODE
  FROM `p-de-sandbox`.temp.assignment_monza_historical  



##DOMUS-150
### Steps
* Run the following code
* Re-run collection_process for all businessdates in prod

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.reference.domus_excluded_tun_ask3423
AS SELECT * FROM `d-cdh`.reference.domus_excluded_tun_ask3423

##DBTC-1178 

In [0]:
%sql
MERGE INTO `p-cdh`.primaryactive.casecode as target
USING `p-cdh`.landed.qc_strategy_step as source
ON target.StrategyStepCode=source.strategy_step_id AND target.BusinessDate=source.BusinessDate AND target.SourceSystem='qc'
WHEN MATCHED THEN 
  UPDATE SET 
            target.ParentStrategyStepcode = source.strategy_id,
            target.ParentStrategyStep = source.strategy_desc,
            target.StrategyTypeCode = source.strategy_area_id,
            target.StrategyType = source.strategy_area_desc

## DBTC-1303

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.primaryarchive.assignment AS 
SELECT * FROM `qa-cdh`.curated.assignmenthist_test

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.derivativearchive.dparty AS 
SELECT * FROM `qa-cdh`.reference.dpartyhist_test

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

